# Taller 1 — Consumo Automatizado de APIs

**Asignatura:** MLY1101 — Machine Learning  
**Nombre del estudiante:** Sebastián Lagos Cantillana  
**Sección:** MLY1101_001V  
**Fecha:** 2026-08-13

## Pregunta u objetivo

> ¿Qué características de calificación, estado de publicación y adaptaciones presentan los mangas registrados en las plataformas AniList, Kitsu y MangaDex?

## Consideraciones generales

- Debe utilizar **3 APIs diferentes** disponibles en: https://github.com/public-apis/public-apis
- Cada API debe aportar información relacionada con el mismo objetivo.
- Debe obtener **mínimo 200 registros por API**, salvo que la fuente disponga de menos registros en total.
- Cada API debe generar un archivo independiente en formato `.json`, `.xlsx`, `.csv` o `.txt`.
- **No realizar merge, join, concat ni cruces entre datasets.**
- El notebook debe poder ejecutarse nuevamente usando **Entorno de ejecución → Ejecutar todas**.


In [33]:
# Librerías base
import requests
import json
import pandas as pd
from pathlib import Path
import time

OUTPUT_DIR = Path('datasets')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Carpeta de salida: {OUTPUT_DIR.resolve()}')

Carpeta de salida: /content/datasets


# Fuente 1 — API 1

**Nombre de la API:** AniList API  
**Documentación:** https://anilist.gitbook.io/anilist-apiv2-docs/  
**Endpoint utilizado:** https://graphql.anilist.co (POST)  
**Descripción de los datos:** Proporciona metadata estructurada sobre mangas, títulos, estado y puntajes promedio mediante GraphQL.  
**Relación con el objetivo:** Permite identificar mangas específicos y extraer su calificación en la plataforma.

In [34]:
# CONFIGURACIÓN API 1
API1_URL = 'https://graphql.anilist.co'
API1_MIN_REGISTROS = 200

api1_headers = {
    'Content-Type': 'application/json',
    'Accept': 'application/json'
}

# La API requiere enviar la estructura de consulta GraphQL
api1_query = '''
query ($page: Int, $perPage: Int) {
  Page(page: $page, perPage: $perPage) {
    media(type: MANGA) {
      id
      title {
        romaji
      }
      status
      averageScore
    }
  }
}
'''

### Paginación API 1

Si la API utiliza paginación, reemplace el bloque anterior por un ciclo que continúe hasta obtener al menos 200 registros o hasta que no existan más páginas.


In [35]:
# CONSUMO API 1 (Paginación integrada)
api1_registros = []
page = 1

print("Extrayendo datos de AniList API...")
while len(api1_registros) < API1_MIN_REGISTROS:
    variables = {'page': page, 'perPage': 50}
    payload = {'query': api1_query, 'variables': variables}

    response = requests.post(API1_URL, headers=api1_headers, json=payload, timeout=30)
    response.raise_for_status()

    data = response.json()
    nuevos = data.get('data', {}).get('Page', {}).get('media', [])

    if not nuevos:
        break

    api1_registros.extend(nuevos)
    page += 1
    time.sleep(1) # Respetar rate limit

print('Status API 1:', response.status_code)
print('Registros API 1:', len(api1_registros))

Extrayendo datos de AniList API...
Status API 1: 200
Registros API 1: 200


In [36]:
# GUARDAR DATASET API 1
API1_ARCHIVO = OUTPUT_DIR / 'dataset_api_1.json'

with open(API1_ARCHIVO, 'w', encoding='utf-8') as f:
    json.dump(api1_registros, f, ensure_ascii=False, indent=2)

print('Archivo generado:', API1_ARCHIVO)
print('Registros guardados:', len(api1_registros))

Archivo generado: datasets/dataset_api_1.json
Registros guardados: 200


# Fuente 2 — API 2

**Nombre de la API:** Kitsu API  
**Documentación:** https://kitsu.docs.apiary.io/  
**Endpoint utilizado:** /edge/manga  
**Descripción de los datos:** Base de datos con sistemas de rating y metadata de publicación.  
**Relación con el objetivo:** Permite contrastar valoraciones y fechas de publicación.

In [37]:
# CONFIGURACIÓN API 2
API2_URL = 'https://kitsu.io/api/edge/manga'
API2_MIN_REGISTROS = 200

api2_headers = {
    'Accept': 'application/vnd.api+json'
}

api2_limit = 20

In [38]:
# CONSUMO API 2
api2_registros = []
offset = 0

print("Extrayendo datos de Kitsu API...")
while len(api2_registros) < API2_MIN_REGISTROS:
    params = {'page[limit]': api2_limit, 'page[offset]': offset}

    response = requests.get(API2_URL, headers=api2_headers, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    nuevos = data.get('data', [])

    if not nuevos:
        break

    api2_registros.extend(nuevos)
    offset += api2_limit

print('Status API 2:', response.status_code)
print('Registros API 2:', len(api2_registros))

Extrayendo datos de Kitsu API...
Status API 2: 200
Registros API 2: 200


In [39]:
# GUARDAR DATASET API 2
API2_ARCHIVO = OUTPUT_DIR / 'dataset_api_2.json'

with open(API2_ARCHIVO, 'w', encoding='utf-8') as f:
    json.dump(api2_registros, f, ensure_ascii=False, indent=2)

print('Archivo generado:', API2_ARCHIVO)
print('Registros guardados:', len(api2_registros))

Archivo generado: datasets/dataset_api_2.json
Registros guardados: 200


# Fuente 3 — API 3

**Nombre de la API:** MangaDex API  
**Documentación:** https://api.mangadex.org/docs/  
**Endpoint utilizado:** /manga  
**Descripción de los datos:** Provee datos técnicos sobre demografía, estado de publicación y géneros.  
**Relación con el objetivo:** Añade variables estructurales que complementan las notas y el estado de publicación.

In [40]:
# CONFIGURACIÓN API 3
API3_URL = 'https://api.mangadex.org/manga'
API3_MIN_REGISTROS = 200

api3_headers = {}
api3_limit = 32

In [41]:
# CONSUMO API 3
api3_registros = []
offset = 0

print("Extrayendo datos de MangaDex API...")
while len(api3_registros) < API3_MIN_REGISTROS:
    params = {'limit': api3_limit, 'offset': offset}

    response = requests.get(API3_URL, headers=api3_headers, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    nuevos = data.get('data', [])

    if not nuevos:
        break

    api3_registros.extend(nuevos)
    offset += api3_limit

print('Status API 3:', response.status_code)
print('Registros API 3:', len(api3_registros))

Extrayendo datos de MangaDex API...
Status API 3: 200
Registros API 3: 224


In [42]:
# GUARDAR DATASET API 3
API3_ARCHIVO = OUTPUT_DIR / 'dataset_api_3.json'

with open(API3_ARCHIVO, 'w', encoding='utf-8') as f:
    json.dump(api3_registros, f, ensure_ascii=False, indent=2)

print('Archivo generado:', API3_ARCHIVO)
print('Registros guardados:', len(api3_registros))

Archivo generado: datasets/dataset_api_3.json
Registros guardados: 224


# Resumen final

La siguiente celda debe ejecutarse al final y mostrar el resultado real de la carga.


In [43]:
total = len(api1_registros) + len(api2_registros) + len(api3_registros)

print('RESUMEN DE CARGA')
print('-' * 50)
print(f'API 1: {len(api1_registros)} registros - {API1_ARCHIVO.name}')
print(f'API 2: {len(api2_registros)} registros - {API2_ARCHIVO.name}')
print(f'API 3: {len(api3_registros)} registros - {API3_ARCHIVO.name}')
print('-' * 50)
print(f'TOTAL: {total} registros')

if len(api1_registros) < 200:
    print('ADVERTENCIA: API 1 tiene menos de 200 registros.')
if len(api2_registros) < 200:
    print('ADVERTENCIA: API 2 tiene menos de 200 registros.')
if len(api3_registros) < 200:
    print('ADVERTENCIA: API 3 tiene menos de 200 registros.')

RESUMEN DE CARGA
--------------------------------------------------
API 1: 200 registros - dataset_api_1.json
API 2: 200 registros - dataset_api_2.json
API 3: 224 registros - dataset_api_3.json
--------------------------------------------------
TOTAL: 624 registros


# Entrega en GitHub

El repositorio debe contener como mínimo:

```text
MLY1101-Taller1-ApellidoNombre/
├── MLY1101_001V_T01_ApellidoNombre.ipynb
├── dataset_api_1.xxx
├── dataset_api_2.xxx
├── dataset_api_3.xxx
└── README.md
```

Si el repositorio es privado, agregar como colaborador a **titiriemann**.


## Checklist final

- [ ] Definí una pregunta u objetivo común.
- [ ] Utilicé 3 APIs del repositorio indicado.
- [ ] Obtuve al menos 200 registros por API o documenté una excepción válida.
- [ ] Generé 3 archivos independientes.
- [ ] No uní ni crucé los datasets.
- [ ] El notebook ejecuta desde cero.
- [ ] Subí notebook, datasets y README.md a GitHub.
- [ ] Entregué la URL del repositorio.
